In [2]:
pip install torch gymnasium numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import numpy as np

GAMMA = 0.99
LAMBDA = 0.95
EPSILON = 0.2
LR = 3e-3
ROLLOUT_STEPS = 500
UPDATES = 150


class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh()
        )

        self.actor = nn.Linear(64, action_dim)
        self.critic = nn.Linear(64, 1)

    def forward(self, x):
        x = self.body(x)
        return self.actor(x), self.critic(x).squeeze(-1)


def compute_gae(rewards, values, next_value, dones):
    values = values + [next_value]

    advantages = [0] * len(rewards)
    gae = 0

    for t in reversed(range(len(rewards))):
        mask = 1 - int(dones[t])

        delta = (
            rewards[t]
            + GAMMA * values[t + 1] * mask
            - values[t]
        )

        gae = delta + GAMMA * LAMBDA * mask * gae
        advantages[t] = gae

    returns = [
        advantage + value
        for advantage, value in zip(advantages, values[:-1])
    ]

    return advantages, returns


def train():

    env = gym.make("CartPole-v1")

    model = ActorCritic(
        env.observation_space.shape[0],
        env.action_space.n
    )

    optimizer = optim.Adam(model.parameters(), lr=LR)

    state, _ = env.reset()

    for update in range(UPDATES):

        states = []
        actions = []
        rewards = []
        dones = []
        old_log_probs = []
        values = []

        for _ in range(ROLLOUT_STEPS):

            s = torch.FloatTensor(state)

            logits, value = model(s)

            dist = Categorical(logits=logits)

            action = dist.sample()

            next_state, reward, terminated, truncated, _ = env.step(
                action.item()
            )

            done = terminated or truncated

            states.append(state)
            actions.append(action.item())
            rewards.append(reward)
            dones.append(done)
            old_log_probs.append(
                dist.log_prob(action).item()
            )
            values.append(value.item())

            state = next_state if not done else env.reset()[0]

        with torch.no_grad():
            _, next_value = model(
                torch.FloatTensor(state)
            )

        advantages, returns = compute_gae(
            rewards,
            values,
            next_value.item(),
            dones
        )

        states_t = torch.FloatTensor(
            np.array(states)
        )

        actions_t = torch.LongTensor(actions)

        old_log_probs_t = torch.FloatTensor(
            old_log_probs
        )

        returns_t = torch.FloatTensor(
            returns
        )

        advantages_t = torch.FloatTensor(
            advantages
        )


        advantages_t = (
            advantages_t - advantages_t.mean()
        ) / (advantages_t.std() + 1e-8)

        logits, values_pred = model(states_t)

        dist = Categorical(logits=logits)

        new_log_probs = dist.log_prob(actions_t)

        entropy = dist.entropy().mean()


        ratio = torch.exp(
            new_log_probs - old_log_probs_t
        )

        surr1 = ratio * advantages_t

        surr2 = torch.clamp(
            ratio,
            1 - EPSILON,
            1 + EPSILON
        ) * advantages_t

        policy_loss = -torch.min(
            surr1,
            surr2
        ).mean()


        value_clipped = returns_t + torch.clamp(
            values_pred - returns_t,
            -EPSILON,
            EPSILON
        )

        value_loss = 0.5 * torch.max(
            (values_pred - returns_t) ** 2,
            (value_clipped - returns_t) ** 2
        ).mean()


        loss = (
            policy_loss
            + 0.5 * value_loss
            - 0.01 * entropy
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        if update % 10 == 0:
            episode_count = max(1, sum(dones))
            avg_reward = sum(rewards) / episode_count

            print(
                f"Update {update} | "
                f"avg reward: {avg_reward:.1f} | "
                f"entropy: {entropy.item():.3f}"
            )

    env.close()



if __name__ == "__main__":
    train()

Update 0 | avg reward: 26.3 | entropy: 0.679
Update 10 | avg reward: 33.3 | entropy: 0.652
Update 20 | avg reward: 55.6 | entropy: 0.637
Update 30 | avg reward: 62.5 | entropy: 0.620
Update 40 | avg reward: 100.0 | entropy: 0.621
Update 50 | avg reward: 83.3 | entropy: 0.617
Update 60 | avg reward: 100.0 | entropy: 0.604
Update 70 | avg reward: 100.0 | entropy: 0.618
Update 80 | avg reward: 83.3 | entropy: 0.621
Update 90 | avg reward: 500.0 | entropy: 0.623
Update 100 | avg reward: 100.0 | entropy: 0.622
Update 110 | avg reward: 83.3 | entropy: 0.626
Update 120 | avg reward: 250.0 | entropy: 0.645
Update 130 | avg reward: 71.4 | entropy: 0.631
Update 140 | avg reward: 55.6 | entropy: 0.628
